# Iris WWTP — Fine-tune YOLO-seg foam (Google Colab)

**Mục tiêu:** train trên Colab (GPU) → tải về `best.pt` → bỏ vào project local:
`Project-monitoring-model/models/foam/best.pt`

**Trước khi chạy:** Runtime → Change runtime type → **GPU (T4)**.

Dataset: [foam ipal](https://universe.roboflow.com/aqua-vision-0whs7/foam-ipal-3dcyf-zy5ul) (CC BY 4.0).

## 1) Cài thư viện

In [ ]:
!pip -q install ultralytics roboflow opencv-python-headless

import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2) Tải dataset từ Roboflow (YOLO Segmentation)

1. Vào [Roboflow Account → API Keys](https://app.roboflow.com/settings/api) lấy key.
2. Dán vào `ROBOFLOW_API_KEY` bên dưới.
3. Chạy cell — tự chọn **version mới nhất** có trên API (đừng hardcode `3` nếu API báo *Version number X is not found*; thử `VERSION = 1`).
4. Colab tải **YOLOv8** format (seg nếu dataset là instance segmentation).

In [ ]:
import os
from pathlib import Path

# === SỬA DÒNG NÀY ===
ROBOFLOW_API_KEY = "PASTE_YOUR_ROBOFLOW_API_KEY_HERE"

assert ROBOFLOW_API_KEY and "PASTE_" not in ROBOFLOW_API_KEY, "Dán API key Roboflow trước"

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
# foam ipal — Aqua Vision
project = rf.workspace("aqua-vision-0whs7").project("foam-ipal-3dcyf-zy5ul")

# Liệt kê version có thật trên API (tránh lỗi Version number X is not found)
versions = list(project.versions())
print("Available versions:", [getattr(v, "version", v) for v in versions])
if not versions:
    raise RuntimeError("Project không có version nào để download")

# Ưu tiên version mới nhất; hoặc set tay VERSION = 1
def _ver_num(v):
    raw = getattr(v, "version", None) or getattr(v, "id", None) or str(v)
    try:
        return int(str(raw).split("/")[-1])
    except Exception:
        return 0

VERSION = max(_ver_num(v) for v in versions)  # hoặc ghi đè: VERSION = 1
print("Using VERSION =", VERSION)
dataset = project.version(VERSION).download("yolov8")

DATA_ROOT = Path(dataset.location)
print("Dataset at:", DATA_ROOT)
print("data.yaml exists:", (DATA_ROOT / "data.yaml").exists())
print(list(DATA_ROOT.iterdir())[:20])

### (Tuỳ chọn) Upload zip COCO từ máy nếu không dùng API

Bỏ qua nếu cell Roboflow ở trên đã OK.  
Nếu đã tải COCO Seg zip: upload lên Colab rồi sửa `ZIP_PATH`.

In [ ]:
# Chỉ chạy nếu KHÔNG dùng Roboflow API ở trên
USE_LOCAL_ZIP = False  # đổi True nếu upload zip COCO

if USE_LOCAL_ZIP:
    import zipfile, json, shutil
    from google.colab import files

    print("Chọn file .zip COCO Segmentation...")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    ZIP_PATH = Path("/content") / zip_name
    COCO_DIR = Path("/content/foam_coco")
    YOLO_DIR = Path("/content/foam_yolo")
    if COCO_DIR.exists():
        shutil.rmtree(COCO_DIR)
    COCO_DIR.mkdir(parents=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(COCO_DIR)

    # Nếu zip bọc 1 folder con, đi vào trong
    kids = [p for p in COCO_DIR.iterdir() if p.is_dir()]
    coco_root = kids[0] if len(kids) == 1 and not (COCO_DIR / "train").exists() else COCO_DIR

    def poly_to_yolo(class_id, segmentation, w, h):
        lines = []
        if not segmentation or isinstance(segmentation, dict):
            return lines
        polys = segmentation if isinstance(segmentation[0], list) else [segmentation]
        for poly in polys:
            if len(poly) < 6:
                continue
            coords = []
            for i in range(0, len(poly), 2):
                coords += [f"{max(0,min(1, float(poly[i])/w)):.6f}", f"{max(0,min(1, float(poly[i+1])/h)):.6f}"]
            lines.append(f"{class_id} " + " ".join(coords))
        return lines

    def convert_split(split_dir, out_root, split_name):
        ann = split_dir / "_annotations.coco.json"
        if not ann.exists():
            js = list(split_dir.glob("*.json"))
            if not js:
                return 0, []
            ann = js[0]
        coco = json.loads(ann.read_text())
        cats = sorted(coco.get("categories", []), key=lambda c: c["id"])
        cat_map = {c["id"]: i for i, c in enumerate(cats)}
        names = [c["name"] for c in cats]
        images = {im["id"]: im for im in coco.get("images", [])}
        anns = {}
        for a in coco.get("annotations", []):
            anns.setdefault(a["image_id"], []).append(a)
        img_out = out_root / "images" / split_name
        lbl_out = out_root / "labels" / split_name
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        n = 0
        for img_id, im in images.items():
            src = split_dir / im["file_name"]
            if not src.exists():
                cands = list(split_dir.glob(f"*{Path(im['file_name']).name}"))
                if not cands:
                    continue
                src = cands[0]
            dest = img_out / src.name
            if not dest.exists():
                shutil.copy2(src, dest)
            w, h = float(im["width"]), float(im["height"])
            lines = []
            for a in anns.get(img_id, []):
                cid = cat_map.get(a["category_id"])
                if cid is None:
                    continue
                lines += poly_to_yolo(cid, a.get("segmentation"), w, h)
            (lbl_out / f"{src.stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
            n += 1
        return n, names

    if YOLO_DIR.exists():
        shutil.rmtree(YOLO_DIR)
    class_names = []
    for folder, split in [("train", "train"), ("valid", "val"), ("val", "val"), ("test", "test")]:
        d = coco_root / folder
        if d.is_dir():
            n, names = convert_split(d, YOLO_DIR, split)
            print(folder, "→", split, n)
            if names:
                class_names = names
    if not class_names:
        class_names = ["foam"]
    val_rel = "images/val" if (YOLO_DIR / "images" / "val").exists() else "images/train"
    yaml = f"path: {YOLO_DIR.as_posix()}\ntrain: images/train\nval: {val_rel}\nnames:\n"
    yaml += "".join(f"  {i}: {n}\n" for i, n in enumerate(class_names))
    (YOLO_DIR / "data.yaml").write_text(yaml)
    DATA_ROOT = YOLO_DIR
    print("DATA_ROOT=", DATA_ROOT)

## 3) Fine-tune YOLOv8n-seg

In [ ]:
from ultralytics import YOLO

DATA_YAML = Path(DATA_ROOT) / "data.yaml"
assert DATA_YAML.exists(), f"Thiếu data.yaml tại {DATA_YAML}"

EPOCHS = 50
IMGSZ = 640
BATCH = 16  # giảm xuống 8 nếu OOM

model = YOLO("yolov8n-seg.pt")  # pretrained COCO-seg
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/content/runs/foam",
    name="seg",
    exist_ok=True,
)

BEST = Path("/content/runs/foam/seg/weights/best.pt")
if not BEST.exists():
    BEST = Path(results.save_dir) / "weights" / "best.pt"
print("best.pt →", BEST, "exists=", BEST.exists())

## 4) Kiểm tra nhanh 1 ảnh

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
from IPython.display import Image, display

best = Path("/content/runs/foam/seg/weights/best.pt")
m = YOLO(str(best))

# lấy 1 ảnh train bất kỳ
sample_imgs = list((Path(DATA_ROOT) / "train" / "images").glob("*.*"))
if not sample_imgs:
    sample_imgs = list(Path(DATA_ROOT).rglob("*.jpg"))[:5]
assert sample_imgs, "Không tìm thấy ảnh mẫu"
img_path = sample_imgs[0]
print("Sample:", img_path)

pred = m.predict(str(img_path), conf=0.25, verbose=False)[0]
out = pred.plot()
out_path = "/content/foam_preview.jpg"
cv2.imwrite(out_path, out)
display(Image(out_path))

# coverage thô trên full frame
if pred.masks is not None:
    h, w = pred.orig_shape
    union = np.zeros((h, w), dtype=np.uint8)
    for mask in pred.masks.data.cpu().numpy():
        msk = cv2.resize(mask, (w, h))
        union[msk > 0.5] = 1
    cov = 100.0 * union.sum() / (h * w)
    print(f"coverage ~ {cov:.2f}%")
else:
    print("Không có mask trên ảnh này (conf thấp hoặc không foam).")

## 5) Tải `best.pt` về máy

Sau khi download, copy vào project:

```text
Project-monitoring-model/models/foam/best.pt
```

`configs/foam.yaml` đã để `backend: yolo` và `weights_path: models/foam/best.pt`.

Local test:

```bat
pip install ultralytics
python scripts/run_foam_coverage.py path\to\frame.jpg
```

In [ ]:
from google.colab import files

best = Path("/content/runs/foam/seg/weights/best.pt")
assert best.exists(), "Chưa có best.pt — chạy cell train trước"
files.download(str(best))
print("Downloaded best.pt — đặt vào models/foam/best.pt trên máy local")

## 6) (Tuỳ chọn) Export ONNX nhẹ hơn cho on-prem

Iris hiện infer bằng Ultralytics `.pt`. Nếu sau này muốn ONNX:

In [ ]:
from ultralytics import YOLO

m = YOLO("/content/runs/foam/seg/weights/best.pt")
onnx_path = m.export(format="onnx", imgsz=640, simplify=True)
print("ONNX:", onnx_path)
from google.colab import files
files.download(onnx_path)